In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("hr_raw.csv")

In [3]:
print("Veri seti boyutu: ", df.shape)

Veri seti boyutu:  (2000000, 15)


In [4]:
df.head()

,Employee_ID,Full_Name,Department,Job_Title,Hire_Date,Performance_Rating,Experience_Years,Status,Work_Mode,Salary,Year,Country,City,Age,Job_Level
0,EMP0000001,Heinz-Georg Eimer,Sales,Business Development,2023-01-31,Satisfactory,8,Active,On-site,92992.0,2023,Germany,Munich,32,Mid
1,EMP0000002,Maartje van den Nuwenhuysen-Geertsen,HR,HR Manager,2008-11-07,Good,11,Active,On-site,112318.0,2008,Netherlands,Eindhoven,43,Senior
2,EMP0000003,Sara Sureda Figueroa,HR,Talent Specialist,2016-03-19,Needs Improvement,15,Active,On-site,111121.0,2016,Spain,Seville,43,Senior
3,EMP0000004,Luce Sanchez,Operations,Operations Director,2024-04-03,Good,1,Active,Hybrid,49012.0,2024,France,Marseille,23,Junior
4,EMP0000005,William Jennings,Sales,Regional Lead,2024-11-17,Good,2,Active,On-site,57553.0,2024,United Kingdom,Edinburgh,31,Junior


In [5]:
df.tail()

,Employee_ID,Full_Name,Department,Job_Title,Hire_Date,Performance_Rating,Experience_Years,Status,Work_Mode,Salary,Year,Country,City,Age,Job_Level
1999995,EMP1999996,Ottmar Kusch-Dörschner,IT,Software Developer,2020-11-30,Good,15,Active,Hybrid,136684.0,2020,Germany,Frankfurt,44,Senior
1999996,EMP1999997,Isis Wagenvoort,Sales,Sales Manager,2024-06-06,Good,3,Active,On-site,57911.0,2024,Netherlands,Amsterdam,25,Junior
1999997,EMP1999998,Ángel Gomez Arenas,Sales,Sales Manager,2020-05-21,Excellent,14,Active,Remote,146065.0,2020,Spain,Valencia,39,Senior
1999998,EMP1999999,Carmela Weiß,Sales,Business Development,2018-07-11,Excellent,9,Active,Remote,138855.0,2018,Germany,Frankfurt,35,Senior
1999999,EMP2000000,Macario Casals,IT,Software Developer,2025-09-11,Good,1,Active,Hybrid,46067.0,2025,Spain,Madrid,28,Junior


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000000 entries, 0 to 1999999
Data columns (total 15 columns):
 #   Column              Dtype  
---  ------              -----  
 0   Employee_ID         object 
 1   Full_Name           object 
 2   Department          object 
 3   Job_Title           object 
 4   Hire_Date           object 
 5   Performance_Rating  object 
 6   Experience_Years    int64  
 7   Status              object 
 8   Work_Mode           object 
 9   Salary              float64
 10  Year                int64  
 11  Country             object 
 12  City                object 
 13  Age                 int64  
 14  Job_Level           object 
dtypes: float64(1), int64(3), object(11)
memory usage: 228.9+ MB


In [7]:
df["Hire_Date"] = pd.to_datetime(df["Hire_Date"])

In [8]:
print(df["Hire_Date"].dtype)

datetime64[ns]


In [9]:
df.isnull().sum()

Employee_ID              0
Full_Name                0
Department               0
Job_Title                0
Hire_Date                0
Performance_Rating    3333
Experience_Years         0
Status                   0
Work_Mode                0
Salary                   0
Year                     0
Country                  0
City                     0
Age                      0
Job_Level                0
dtype: int64

In [10]:
df["Performance_Rating"].value_counts(dropna=False)

Performance_Rating
Good                 998968
Satisfactory         558931
Excellent            298844
Needs Improvement    139924
NaN                    3333
Name: count, dtype: int64

In [11]:
df_filled = df.copy()

In [12]:
target_col = "Performance_Rating"
stratify_by = ["Job_Level", "Department"]
for groups, group_df in df.groupby(stratify_by):
    observed_values = group_df[target_col].dropna()
    missing_indices = group_df[group_df[target_col].isnull()].index

    if len(missing_indices) > 0 and not observed_values.empty:
        sampled_values = np.random.choice(observed_values, size=len(missing_indices))
        df_filled.loc[missing_indices, target_col] = sampled_values

In [13]:
df = df_filled.copy()

In [14]:
print(f"Kalan Eksik Veri: {df['Performance_Rating'].isnull().sum()}")

Kalan Eksik Veri: 0


In [15]:
df.isnull().sum()

Employee_ID           0
Full_Name             0
Department            0
Job_Title             0
Hire_Date             0
Performance_Rating    0
Experience_Years      0
Status                0
Work_Mode             0
Salary                0
Year                  0
Country               0
City                  0
Age                   0
Job_Level             0
dtype: int64

In [16]:
df["Salary"].max()

322107.0

In [17]:
df["Salary"].min()

-99932.0

In [18]:
df.loc[df["Salary"] <= 0, "Salary"] = np.nan

In [19]:
df["Salary"] = df["Salary"].fillna(
    df.groupby(["Job_Level", "Department"])["Salary"].transform("median")
)

In [20]:
df["Salary"] = df["Salary"].fillna(df["Salary"].median())

In [21]:
print("Salary Distribution Summary :")
df["Salary"].describe()

Salary Distribution Summary :


count    2.000000e+06
mean     9.018084e+04
std      4.656802e+04
min      3.330800e+04
25%      5.378000e+04
50%      8.090650e+04
75%      1.042710e+05
max      3.221070e+05
Name: Salary, dtype: float64

In [22]:
print("Average Salary per Job Level")
df.groupby("Job_Level")["Salary"].mean().sort_values()

Average Salary per Job Level


Job_Level
Junior       50758.425509
Mid          86103.897456
Senior      141102.651503
Director    226032.652864
Name: Salary, dtype: float64

In [23]:
df["Experience_Years"]

0           8
1          11
2          15
3           1
4           2
           ..
1999995    15
1999996     3
1999997    14
1999998     9
1999999     1
Name: Experience_Years, Length: 2000000, dtype: int64

In [24]:
df[df["Experience_Years"] < 0].groupby("Experience_Years").size()

Experience_Years
-5    485
-4    491
-3    473
-2    504
-1    468
dtype: int64

In [25]:
df.loc[df['Experience_Years'] < 0, 'Experience_Years'] = np.nan

In [26]:
df["Experience_Years"] = df["Experience_Years"].fillna(
    df.groupby("Job_Level")["Experience_Years"].transform("median")
)

In [27]:
df["Experience_Years"] = df["Experience_Years"].round(0).astype(int)

In [28]:
print("Negatif değerlerin kontrolü : ")
(df["Experience_Years"] < 0).sum()

Negatif değerlerin kontrolü : 


np.int64(0)

In [29]:
print("Deneyim yıllarının Özeti : ")
df.groupby("Job_Level")["Experience_Years"].agg(["min", "max", "median"])

Deneyim yıllarının Özeti : 


,min,max,median
Job_Level,,,
Director,0,30,23.0
Junior,0,3,2.0
Mid,0,8,6.0
Senior,0,15,12.0


In [30]:
advanced_levels = ["Mid", "Senior", "Director"]
mask = (df["Job_Level"].isin(advanced_levels)) & (df["Experience_Years"] == 0)
df.loc[mask, "Experience_Years"] = np.nan

In [31]:
df["Experience_Years"] = df["Experience_Years"].fillna(
    df.groupby("Job_Level")["Experience_Years"].transform("median")
)

In [32]:
print("Düzeltilmiş Deneyim yılları : ")
summary = df.groupby("Job_Level")["Experience_Years"].agg(["min", "max", "median"])
summary

Düzeltilmiş Deneyim yılları : 


,min,max,median
Job_Level,,,
Director,1.0,30.0,23.0
Junior,0.0,3.0,2.0
Mid,1.0,8.0,6.0
Senior,1.0,15.0,12.0


In [33]:
experience_1_years = df[df["Experience_Years"] == 1]

In [34]:
countByLevel = experience_1_years.groupby("Job_Level").size().reset_index(name="Count_of_1_Years_Exp")

In [35]:
print("İş seviyesine göre 1 yıllık deneyim sıklığı: ")
countByLevel

İş seviyesine göre 1 yıllık deneyim sıklığı: 


,Job_Level,Count_of_1_Years_Exp
0,Director,24
1,Junior,189527
2,Mid,185
3,Senior,84


In [36]:
total_per_level = df.groupby("Job_Level").size()
percentage = (experience_1_years.groupby("Job_Level").size() / total_per_level * 100).round(2)

In [37]:
print("Her kategoriye göre 1 yıllık deneyim kayıtlarının yüzdesi: ")
percentage

Her kategoriye göre 1 yıllık deneyim kayıtlarının yüzdesi: 


Job_Level
Director     0.02
Junior      25.00
Mid          0.02
Senior       0.02
dtype: float64

In [38]:
logicFloor = {
    "Mid": 3, # min 3 yıl
    "Senior": 7, # min 7 yıl
    "Director": 15 # min 15 yıl
}

In [39]:
for level, min_exp in logicFloor.items():
    mask = (df["Job_Level"] == level) & (df["Experience_Years"] < min_exp)
    df.loc[mask, "Experience_Years"] = np.nan

In [40]:
df["Experience_Years"] = df["Experience_Years"].fillna(
    df.groupby("Job_Level")["Experience_Years"].transform("median")
)

In [41]:
df.groupby('Job_Level')['Experience_Years'].agg(['min', 'max', 'median'])

,min,max,median
Job_Level,,,
Director,16.0,30.0,23.0
Junior,0.0,3.0,2.0
Mid,4.0,8.0,6.0
Senior,9.0,15.0,12.0


In [42]:
min_graduation_age = 21
invalid_age_mask = df["Age"] < (df["Experience_Years"] + min_graduation_age)

In [43]:
print(f"Mantıksız yaş kayıtları : {invalid_age_mask.sum()}")

Mantıksız yaş kayıtları : 48584


In [44]:
df.loc[invalid_age_mask,"Age"] = df["Experience_Years"] + 22
df["Age"] = df["Age"].astype(int)

In [45]:
print("İş Seviyesine Göre Yaş Özeti (Düzeltme Sonrası): ")
df.groupby("Job_Level")["Age"].agg(["min", "max", "mean"]).round(1)

İş Seviyesine Göre Yaş Özeti (Düzeltme Sonrası): 


,min,max,mean
Job_Level,,,
Director,37,66,50.0
Junior,21,39,26.2
Mid,25,44,31.0
Senior,30,51,37.8


In [46]:
print("Kişi sayısı: ")
df["Status"].value_counts(dropna=False)

Kişi sayısı: 


Status
Active        1793596
Resigned       160000
Terminated      45951
Retired           453
Name: count, dtype: int64

In [47]:
print("\nYüzdelik Oran (%):")
df["Status"].value_counts(normalize=True)* 100


Yüzdelik Oran (%):


Status
Active        89.67980
Resigned       8.00000
Terminated     2.29755
Retired        0.02265
Name: proportion, dtype: float64

In [48]:
df = df[df["Status"].isin(["Active", "Resigned"])].copy()

In [49]:
status_map = {
    "Resigned": 1,
    "Active": 0
}
df["Status"] = df["Status"].map(status_map)

In [50]:
df["Status"].value_counts(dropna=False)

Status
0    1793596
1     160000
Name: count, dtype: int64

In [51]:
print("\nYeni Yüzdelik Oran (%):")
df["Status"].value_counts(normalize=True) * 100


Yeni Yüzdelik Oran (%):


Status
0    91.809975
1     8.190025
Name: proportion, dtype: float64

In [52]:
# Job_Level için Ordinal Encoding
kidem_sozlugu = {
    "Junior":1,
    "Mid":2,
    "Senior":3,
    "Director":4
}

if df["Job_Level"].dtype == "object":
    df["Job_Level"] = df["Job_Level"].map(kidem_sozlugu)

In [53]:
silinecek_sutunlar = ['Employee_ID', 'Hire_Date']
df = df.drop(columns=silinecek_sutunlar, errors='ignore')
df = pd.get_dummies(df, columns=['Department'], drop_first=True)

In [54]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1953596 entries, 0 to 1999999
Data columns (total 16 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   Full_Name              object 
 1   Job_Title              object 
 2   Performance_Rating     object 
 3   Experience_Years       float64
 4   Status                 int64  
 5   Work_Mode              object 
 6   Salary                 float64
 7   Year                   int64  
 8   Country                object 
 9   City                   object 
 10  Age                    int64  
 11  Job_Level              int64  
 12  Department_HR          bool   
 13  Department_IT          bool   
 14  Department_Operations  bool   
 15  Department_Sales       bool   
dtypes: bool(4), float64(2), int64(4), object(6)
memory usage: 201.2+ MB


In [55]:
df = df.drop(columns=['Full_Name'], errors='ignore')

print("--- KATEGORİK SÜTUNLARIN RÖNTGENİ ---")
print(f"Performance_Rating Benzersiz Değerler: {df['Performance_Rating'].unique()}")
print(f"Work_Mode Benzersiz Değerler: {df['Work_Mode'].unique()}")
print(f"Job_Title Benzersiz Kayıt Sayısı: {df['Job_Title'].nunique()}")
print(f"Country Benzersiz Kayıt Sayısı: {df['Country'].nunique()}")
print(f"City Benzersiz Kayıt Sayısı: {df['City'].nunique()}")

--- KATEGORİK SÜTUNLARIN RÖNTGENİ ---
Performance_Rating Benzersiz Değerler: ['Satisfactory' 'Good' 'Needs Improvement' 'Excellent']
Work_Mode Benzersiz Değerler: ['On-site' 'Hybrid' 'Remote']
Job_Title Benzersiz Kayıt Sayısı: 25
Country Benzersiz Kayıt Sayısı: 7
City Benzersiz Kayıt Sayısı: 35


In [56]:
perf_sozlugu = {
    'Needs Improvement': 1,
    'Satisfactory': 2,
    'Good': 3,
    'Excellent': 4
}
df['Performance_Rating'] = df['Performance_Rating'].map(perf_sozlugu)

In [57]:
kalan_kategorikler = ['Work_Mode', 'Job_Title', 'Country', 'City']
df = pd.get_dummies(df, columns=kalan_kategorikler, drop_first=True)

In [58]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1953596 entries, 0 to 1999999
Data columns (total 77 columns):
 #   Column                          Dtype  
---  ------                          -----  
 0   Performance_Rating              int64  
 1   Experience_Years                float64
 2   Status                          int64  
 3   Salary                          float64
 4   Year                            int64  
 5   Age                             int64  
 6   Job_Level                       int64  
 7   Department_HR                   bool   
 8   Department_IT                   bool   
 9   Department_Operations           bool   
 10  Department_Sales                bool   
 11  Work_Mode_On-site               bool   
 12  Work_Mode_Remote                bool   
 13  Job_Title_Business Development  bool   
 14  Job_Title_CFO                   bool   
 15  Job_Title_Data Engineer         bool   
 16  Job_Title_DevOps Engineer       bool   
 17  Job_Title_Finance Manager       

In [59]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

In [60]:
X = df.drop(columns =["Status"])
y = df["Status"]

In [61]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [62]:
print(f"Eğitim Verisi: {X_train.shape[0]} satır")
print(f"Test (Sınav) Verisi: {X_test.shape[0]} satır\n")

Eğitim Verisi: 1562876 satır
Test (Sınav) Verisi: 390720 satır



In [63]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight="balanced", n_jobs=-1)

In [64]:
rf_model.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [65]:
y_pred = rf_model.predict(X_test)

In [66]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.92      1.00      0.96    358720
           1       0.15      0.01      0.02     32000

    accuracy                           0.92    390720
   macro avg       0.54      0.50      0.49    390720
weighted avg       0.86      0.92      0.88    390720



In [67]:
from sklearn.metrics import confusion_matrix

In [68]:
cm = confusion_matrix(y_test, y_pred)
print("--- KARMAŞIKLIK MATRİSİ (CONFUSION MATRIX) ---")
print(f"Gerçekte Kalan ve 'Kalır' dediklerimiz (Doğru): {cm[0][0]}")
print(f"Gerçekte Kalan ama 'İstifa Eder' dediklerimiz (Yanlış Alarm): {cm[0][1]}")
print(f"Gerçekte İstifa Eden ama 'Kalır' dediklerimiz (Kaçanlar - FİASKO): {cm[1][0]}")
print(f"Gerçekte İstifa Eden ve 'İstifa Eder' dediklerimiz (Doğru Yakalanan): {cm[1][1]}")

--- KARMAŞIKLIK MATRİSİ (CONFUSION MATRIX) ---
Gerçekte Kalan ve 'Kalır' dediklerimiz (Doğru): 357261
Gerçekte Kalan ama 'İstifa Eder' dediklerimiz (Yanlış Alarm): 1459
Gerçekte İstifa Eden ama 'Kalır' dediklerimiz (Kaçanlar - FİASKO): 31739
Gerçekte İstifa Eden ve 'İstifa Eder' dediklerimiz (Doğru Yakalanan): 261


In [69]:
onem_dereceleri = pd.DataFrame({
    'Özellik': X_train.columns,
    'Önem_Skoru': rf_model.feature_importances_
}).sort_values(by='Önem_Skoru', ascending=False)

In [70]:
print("\n--- MODELİN EN ÇOK DİKKATE ALDIĞI İLK 10 ÖZELLİK ---")
print(onem_dereceleri.head(10))


--- MODELİN EN ÇOK DİKKATE ALDIĞI İLK 10 ÖZELLİK ---
                   Özellik  Önem_Skoru
2                   Salary    0.282707
4                      Age    0.169777
3                     Year    0.126184
1         Experience_Years    0.092178
0       Performance_Rating    0.053605
5                Job_Level    0.025738
10       Work_Mode_On-site    0.019478
11        Work_Mode_Remote    0.015451
41  Country_United Kingdom    0.005643
36         Country_Germany    0.005461


In [71]:
df = df.drop(columns=['Year'], errors='ignore')

In [72]:
istifa_edenler = df[df['Status'] == 1]
kalanlar = df[df['Status'] == 0]

In [73]:
kalanlar_alt_ornek = kalanlar.sample(n=len(istifa_edenler), random_state=42)

In [74]:
df_dengeli = pd.concat([istifa_edenler, kalanlar_alt_ornek])

In [75]:
print(f"Sınıf Dağılımı Artık Kusursuz:\n{df_dengeli['Status'].value_counts()}\n")

Sınıf Dağılımı Artık Kusursuz:
Status
1    160000
0    160000
Name: count, dtype: int64



In [76]:
X_yeni = df_dengeli.drop(columns=['Status'])
y_yeni = df_dengeli['Status']

In [77]:
X_train2, X_test2, y_train2, y_test2 = train_test_split(X_yeni, y_yeni, test_size=0.20, random_state=42)

In [78]:
rf_model_dengeli = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model_dengeli.fit(X_train2, y_train2)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [79]:
y_pred2 = rf_model_dengeli.predict(X_test2)

In [80]:
print("\n--- YENİ (DENGELİ) MODEL ---")
print(classification_report(y_test2, y_pred2))


--- YENİ (DENGELİ) MODEL ---
              precision    recall  f1-score   support

           0       0.61      0.61      0.61     31934
           1       0.61      0.61      0.61     32066

    accuracy                           0.61     64000
   macro avg       0.61      0.61      0.61     64000
weighted avg       0.61      0.61      0.61     64000

